# 05 - Traducción de Negocio, Demostración del Producto y Análisis Ético
**Proyecto:** K-asar — Stellar Data Clustering
**Responsable:** Yohanna
**Fase anterior:** `04_modeling.ipynb` (Josema) — entrenó y comparó K-Means, DBSCAN, GMM y Clustering Jerárquico.

Este es el notebook de cierre del proyecto. Toma el modelo ya entrenado y comparado en `04_modeling.ipynb` y responde a la pregunta que realmente le importa al cliente — Meridian Sky Survey Consortium: **¿esto sirve para algo, es justo, y cómo se usaría mañana por la mañana?**

**Contenido de este notebook:**
1. Recapitulación autosuficiente del pipeline (para poder ejecutar este notebook de forma independiente)
2. Validación honesta: comparación de los 4 modelos contra la clase real
3. Traducción de los 3 clusters de K-Means a categorías de triage de negocio
4. Demo de producto: función de scoring para objetos nuevos
5. Análisis ético
6. Extrapolación a otros dominios y situaciones similares


---
## 0. Recapitulación del pipeline (Notebooks 01–04)

Para que este notebook funcione de forma **autónoma**, sin depender de que los notebooks anteriores ya se hayan ejecutado en la misma sesión, reproducimos aquí de forma condensada cada paso ya justificado en detalle en su notebook correspondiente.

In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, classification_report, accuracy_score
from scipy.optimize import linear_sum_assignment

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Entorno listo.")


In [ ]:
# --- Notebook 01 (Javi): carga y columnas ---
dataset_dir = kagglehub.dataset_download("fedesoriano/stellar-classification-dataset-sdss17")
file_path = os.path.join(dataset_dir, "star_classification.csv")
df_raw = pd.read_csv(file_path)

metadata_cols = ['obj_ID', 'alpha', 'delta', 'run_ID', 'rerun_ID',
                  'cam_col', 'field_ID', 'spec_obj_ID', 'plate', 'MJD', 'fiber_ID']
df = df_raw.drop(columns=metadata_cols)
y_true = df['class'].copy()
X = df.drop(columns=['class'])
print(f"Datos cargados: {df_raw.shape[0]:,} objetos. Columna 'class' retirada del modelado.")


In [ ]:
# --- Notebook 02 (Luis): preprocesamiento ---
magnitude_cols = ['u', 'g', 'r', 'i', 'z']
mask_invalid = (X[magnitude_cols] <= 0).any(axis=1)
X = X[~mask_invalid].reset_index(drop=True)
y_true = y_true[~mask_invalid].reset_index(drop=True)

X_features = X.copy()
X_features['color_ug'] = X_features['u'] - X_features['g']
X_features['color_gr'] = X_features['g'] - X_features['r']
X_features['color_ri'] = X_features['r'] - X_features['i']
X_features['color_iz'] = X_features['i'] - X_features['z']

scaler = RobustScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_features), columns=X_features.columns)
print("Features listas:", X_scaled.shape, "| columnas:", list(X_scaled.columns))


In [ ]:
# --- Notebook 03 (Isabella) + 04 (Josema): PCA y modelo final ---
# El notebook de modelado (04) reoptimizó el umbral de varianza al 95% (5 componentes)
# en lugar de las 4 componentes exploradas al 90% en el notebook 03, por ofrecer
# una separación ligeramente mejor entre clusters.
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA: {pca.n_components_} componentes ({pca.explained_variance_ratio_.sum()*100:.2f}% varianza retenida)")

kmeans_final = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_pca)

X_features['cluster'] = cluster_labels
X_features['class_real'] = y_true.values

print(pd.Series(cluster_labels).value_counts().sort_index().rename("objetos por cluster"))


---
## 1. El momento de la verdad: ¿qué modelo recupera mejor la astrofísica real?

En `04_modeling.ipynb`, Josema entrenó y comparó cuatro algoritmos —K-Means, DBSCAN, GMM y Clustering Jerárquico— contra la clase real (`class`), que en el archivo histórico del survey se obtuvo con espectroscopía cara. En ningún momento del entrenamiento se usó esa columna; solo se usa acá, al final, para validar.

In [ ]:
resultados_comparativa = {
    "K-Means":               {"ARI": 0.2716, "NMI": 0.2858, "Accuracy": 0.6616, "F1_macro": 0.66, "F1_weighted": 0.67},
    "DBSCAN":                {"ARI": 0.0060, "NMI": 0.0037, "Accuracy": 0.5925, "F1_macro": 0.19, "F1_weighted": 0.44},
    "GMM":                   {"ARI": 0.2159, "NMI": 0.3160, "Accuracy": 0.6295, "F1_macro": 0.61, "F1_weighted": 0.64},
    "Clustering Jerárquico": {"ARI": 0.2834, "NMI": 0.2689, "Accuracy": 0.6149, "F1_macro": 0.57, "F1_weighted": 0.63},
}
tabla_comparativa = pd.DataFrame(resultados_comparativa).T
tabla_comparativa


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metrics = ["ARI", "Accuracy", "F1_weighted"]
colors = ["#9163F5", "#4FD8E8", "#E8B34D", "#E0637A"]
for ax, metric in zip(axes, metrics):
    tabla_comparativa[metric].plot(kind="bar", ax=ax, color=colors)
    ax.set_title(metric)
    ax.set_xticklabels(tabla_comparativa.index, rotation=30, ha="right")
    ax.set_ylim(0, 1 if metric != "ARI" else 0.35)
plt.suptitle("Comparación de los 4 modelos contra la clase real", y=1.03)
plt.tight_layout()
plt.show()


**Lectura honesta de esta comparación:**

- **DBSCAN colapsó**: con los hiperparámetros probados (`eps=1, min_samples=5`), casi todo el dataset terminó agrupado en una única megaclase de tipo GALAXY (recall 100%, pero 0% en QSO y STAR). Es la razón por la que su ARI es prácticamente cero — no encontró estructura real, encontró una sola bola densa.
- **Clustering Jerárquico** tiene el ARI más alto (0.2834), pero un F1 muy pobre en STAR (0.31) — separa bien galaxias y cuásares, pero confunde sistemáticamente estrellas.
- **GMM** tiene la mejor NMI (0.3160) y la mejor precisión en GALAXY (0.99) — cuando dice "es una galaxia", casi nunca se equivoca — pero a costa de mezclar mucho QSO y STAR entre sí.
- **K-Means** es el que ofrece el mejor equilibrio entre las tres clases, con el mejor F1 global (0.67 ponderado) y, en particular, el mejor desempeño identificando cuásares (F1=0.81, recall=89%) — justamente el objeto científicamente más valioso y más raro del dataset.

Por este equilibrio, **K-Means (K=3) es el modelo que llevamos a producción** para el resto de este notebook.


In [ ]:
ct = pd.crosstab(X_features["cluster"], X_features["class_real"])
ct_pct = (ct.T / ct.sum(axis=1)).T.round(4)
print("Conteo por cluster x clase real:")
print(ct)
print("\nComposición porcentual de cada cluster:")
print(ct_pct)


In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ct_pct.plot(kind="bar", stacked=True, ax=ax, color=["#9163F5","#E8B34D","#4FD8E8"])
ax.set_xlabel("Cluster (descubierto sin usar 'class')")
ax.set_ylabel("Proporción de cada clase real dentro del cluster")
ax.set_title("Composición real de cada cluster de K-Means")
ax.legend(title="Clase real", bbox_to_anchor=(1.02,1), loc="upper left")
plt.tight_layout()
plt.show()


**Composición real de los 3 clusters de K-Means:**

| Cluster | Tamaño | GALAXY | QSO | STAR | Perfil dominante |
|---|---|---|---|---|---|
| 0 | 44,289 (44.3%) | 82.2% | 2.8% | 15.0% | Galaxia de alta confianza |
| 1 | 22,798 (22.8%) | 17.1% | 73.9% | 9.0% | Candidato QSO |
| 2 | 32,912 (32.9%) | 58.2% | 2.6% | 39.2% | Mezcla galaxia/estrella |

Un **ARI de 0.2716** contra la clase real indica una concordancia **real pero moderada** — no un clasificador perfecto. Y esto es exactamente lo esperable: la fotometría por sí sola es *parcialmente* informativa, pero no reemplaza la espectroscopía. El producto correcto para el cliente **no es "clasificador automático"**, es una **herramienta de triage que prioriza qué observar primero**.


---
## 2. Traduciendo los 3 clusters a categorías de negocio

Basado en la composición real de cada cluster, traducimos los 3 clusters matemáticos en 3 categorías de triage operativo para el equipo de asignación de tiempo de telescopio del consorcio.

In [ ]:
profile_cols = ['u','g','r','i','z','redshift','color_ug','color_gr','color_ri','color_iz']
cluster_profile = X_features.groupby("cluster")[profile_cols].mean().round(3)
cluster_profile["n_objetos"] = X_features.groupby("cluster").size()
cluster_profile


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col in zip(axes, ["redshift","color_ug","color_gr"]):
    sns.barplot(x=cluster_profile.index, y=cluster_profile[col], ax=ax, palette="mako")
    ax.set_title(f"Promedio de {col} por cluster")
    ax.set_xlabel("Cluster")
plt.suptitle("Perfil fotométrico por cluster", y=1.03)
plt.tight_layout()
plt.show()


**Las tres categorías de triage propuestas al cliente:**

- **"Galaxia de alta confianza"** (Cluster 0, 44.3% de los objetos) — 82.2% de pureza real hacia galaxias. *Acción:* prioridad **estándar** — la fotometría sola ya da confianza suficiente, no urge gastar espectroscopía acá.
- **"Candidato QSO — prioridad alta"** (Cluster 1, 22.8%) — mayor concentración relativa de cuásares reales del dataset (73.9%). Los cuásares son los objetos científicamente más raros y valiosos. *Acción:* prioridad **alta** — es donde más rentable es gastar tiempo de telescopio, porque la fotometría sola es menos concluyente y el valor de confirmar un QSO es mayor.
- **"Mezcla galaxia/estrella — revisión estándar"** (Cluster 2, 32.9%) — composición intermedia (58.2% galaxia, 39.2% estrella). *Acción:* prioridad **media**, útil para lotes de seguimiento rutinario.


---
## 3. Demo de producto: función de triage para objetos nuevos

El entregable real para Meridian Sky Survey no es este análisis puntual — es una función reutilizable que el pipeline de reducción de datos del survey pueda invocar automáticamente la misma noche en que se fotografía un objeto nuevo, antes de que nadie gaste tiempo de telescopio en él. Reproduce exactamente el pipeline de escalado (`RobustScaler`) + PCA (95% varianza, 5 componentes) + K-Means ya entrenado.

In [ ]:
FEATURE_ORDER = ['u', 'g', 'r', 'i', 'z', 'redshift', 'color_ug', 'color_gr', 'color_ri', 'color_iz']

PRIORITY_MAP = {
    0: "Prioridad ESTÁNDAR — probable galaxia, fotometría ya suficientemente concluyente",
    1: "Prioridad ALTA — candidato a cuásar (QSO), objeto científicamente valioso y poco concluyente por fotometría sola",
    2: "Prioridad MEDIA — mezcla galaxia/estrella, revisión rutinaria por lotes",
}

def triage_new_object(u, g, r, i, z, redshift):
    """
    Recibe las 5 magnitudes fotométricas y el redshift de un objeto recién
    observado, y devuelve el cluster asignado más una recomendación de
    prioridad de seguimiento espectroscópico. Reproduce el pipeline de
    escalado + PCA + K-Means entrenado en 04_modeling.ipynb.
    """
    row = pd.DataFrame([{"u": u, "g": g, "r": r, "i": i, "z": z, "redshift": redshift}])
    row["color_ug"] = row["u"] - row["g"]
    row["color_gr"] = row["g"] - row["r"]
    row["color_ri"] = row["r"] - row["i"]
    row["color_iz"] = row["i"] - row["z"]

    row_scaled = scaler.transform(row[FEATURE_ORDER])
    row_pca = pca.transform(row_scaled)
    cluster = int(kmeans_final.predict(row_pca)[0])

    return {"cluster": cluster, "recomendacion": PRIORITY_MAP.get(cluster, "Revisar manualmente")}


# DEMO: triage de un objeto recién fotografiado con perfil típico de cuásar
demo_object = {"u": 22.0, "g": 21.2, "r": 20.9, "i": 20.7, "z": 20.6, "redshift": 1.8}
triage_new_object(**demo_object)


- Misma lógica de ingeniería de color, escalado y PCA ajustada en los notebooks 02–04 — sin desalineación entre el análisis y la operación real.
- La salida es directamente accionable por el equipo de planificación de observaciones: una etiqueta de prioridad, no solo un número de cluster.
- Esta función es la semilla de un servicio de triage automático que correría cada noche sobre los nuevos objetos fotografiados por el survey — la versión desplegada en producción puede probarse en vivo en la app de K-asar.


---
## 4. Análisis ético

Automatizar la priorización de qué objetos observar con espectroscopía no es una decisión neutral — moldea qué descubre la ciencia y qué se queda sin observar. Puntos que el equipo debe documentar y monitorear antes de cualquier despliegue real:

- **Sesgo de descubrimiento:** si el modelo de triage se basa en los patrones de color de los objetos ya conocidos, es sistemáticamente menos capaz de reconocer objetos genuinamente nuevos o atípicos — precisamente el tipo de hallazgo más valioso en astronomía. Un sistema de triage mal calibrado podría, sin querer, filtrar hacia abajo en la cola de prioridad exactamente los descubrimientos más interesantes.
- **Sesgo instrumental:** en el notebook 02, Luis descartó deliberadamente todo lo relacionado con el instrumento de observación (`run_ID`, `cam_col`, `field_ID`, etc.) precisamente para evitar que el modelo aprenda patrones del telescopio/cámara en lugar de patrones del objeto. Si en el futuro se añaden más variables, hay que repetir este mismo escrutinio.
- **Equidad entre equipos de investigación:** si distintos grupos compiten por el mismo tiempo de telescopio, una herramienta de triage que sistemáticamente favorezca cierto tipo de objetos (por ejemplo, siempre QSOs, dado que son "prioridad alta") puede desplazar injustamente otras líneas de investigación igual de legítimas (por ejemplo, estudios estadísticos de galaxias comunes).
- **Desempeño desigual entre clases:** la comparación de la Sección 1 mostró que **STAR es sistemáticamente la clase peor identificada** en los cuatro modelos (F1 entre 0.31 y 0.50). Cualquier despliegue debe comunicar con claridad que el triage es menos confiable para candidatos a estrella, y no tratar sus recomendaciones con la misma confianza que las de GALAXY o QSO.
- **Falta de verdad de referencia para casos nuevos:** el ARI de la Sección 1 se calculó contra el archivo histórico, donde ya existe la clase real. Para objetos genuinamente nuevos, no hay forma de confirmar la calidad del triage hasta que, precisamente, se gaste tiempo de espectroscopía en algunos de ellos.

**Salvaguardas recomendadas al cliente:**
1. Usar el triage solo para **ordenar una cola de prioridad**, nunca para descartar objetos por completo sin revisión humana.
2. Reservar una fracción fija del tiempo de telescopio para objetos fuera del top de prioridad, específicamente para detectar si el modelo está sistemáticamente ignorando una clase de descubrimientos.
3. Comunicar explícitamente la menor confiabilidad del modelo para candidatos a estrella.
4. Re-entrenar y re-validar el ARI/NMI contra la clase real cada vez que se acumule un nuevo lote de confirmaciones espectroscópicas.
5. Publicar internamente el perfil de cada cluster (Sección 2) para que la priorización sea auditable por el comité científico del survey.


---
## 5. Extrapolando el enfoque a otros dominios

El pipeline construido por el equipo — retirar la variable objetivo antes de modelar, distinguir errores de instrumento de outliers reales, escalar de forma robusta, reducir dimensionalidad, comparar varios algoritmos de clustering, validar honestamente contra verdad de referencia cuando existe, traducir a categorías accionables, y someter el resultado a un análisis ético — es igual de válido fuera de la astronomía. Solo cambian los datos crudos y qué cuenta como "recurso caro" a optimizar:

- **Salud:** triage de qué pacientes o muestras necesitan una prueba diagnóstica cara (biopsia, resonancia), usando solo señales baratas (síntomas, análisis de sangre básicos).
- **Manufactura:** priorizar qué lotes de producción necesitan una inspección de calidad costosa, usando solo sensores baratos de línea de producción.
- **Ciberseguridad:** priorizar qué alertas de seguridad merecen la revisión cara de un analista senior, usando señales automáticas baratas para un primer filtrado no supervisado.

## 6. Situaciones de negocio similares

- Un banco quiere decidir qué solicitudes de crédito requieren una revisión manual completa, usando solo variables baratas y ya disponibles para un primer filtro.
- Una aerolínea quiere decidir qué motores requieren una inspección de mantenimiento completa antes de lo programado, usando solo telemetría barata de vuelo.
- Un equipo de soporte técnico quiere priorizar qué tickets requieren un ingeniero senior, usando solo el texto del ticket para un triage automático inicial.

En todos los casos, la habilidad transferible no es "K-Means sobre magnitudes fotométricas" — es la disciplina completa: excluir explícitamente cualquier variable que dependa del recurso caro que se busca ahorrar, comparar más de un algoritmo antes de elegir, validar honestamente contra la verdad cuando esté disponible sin adornar el resultado, y acompañar cada segmento con una acción de negocio y una mirada crítica sobre a quién podría perjudicar si el modelo se equivoca.
